# Lab 9: Voluntary "warm-up" and twitch tension
Results / analysis for the proposal *Question 4, lab 4; effects of warming up on muscle contraction*.

1. Histograms of twitch tension: Trial 1, Trial 2, Trial 3 (voluntary 5 cm / 20 s) vs. involuntary (tetanus).
2. Naive OLS of twitch-tension increase vs. **intensity** (distance) and vs. **duration**, no processing. Print average L2 distance.
3. Drop the fatigue point, redo OLS. Print average L2 distance.
4. Fit all points to an exponential instead. Print L2 distance.

> **The numbers below are made-up placeholders** so the notebook runs end to end. Replace them in the *Data* section with your measured values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## Data (replace the placeholders)

### Part 1: single-twitch amplitudes (one value per twitch, same units as your recording, e.g. mV or N)
Each list holds the twitch amplitudes measured **after** the warm-up phase. `involuntary` is the post-tetanic potentiation data from Lab 4 (1 Hz / 6 A twitches after tetanus).

In [ ]:
# PLACEHOLDER DATA: replace with measured twitch tension amplitudes
rng = np.random.default_rng(0)
trial_1     = rng.normal(1.30, 0.10, 20)
trial_2     = rng.normal(1.35, 0.10, 20)
trial_3     = rng.normal(1.28, 0.10, 20)
involuntary = rng.normal(1.60, 0.12, 20)   # post-tetanic potentiation, Lab 4

# optional: baseline twitches before the warm-up (set to None to skip)
baseline = rng.normal(1.00, 0.08, 20)

### Part 2: intensity and duration sweeps
For each condition, `y` is the **increase in twitch amplitude** (after warm-up minus before warm-up).
- Intensity: distance moved, 2, 3, 5, 7, 9, 10 cm (duration fixed at 20 s)
- Duration: 3, 5, 10, 20, 40, 60, 90 s (distance fixed at 5 cm)

The last point in each sweep is the expected **fatigue point** (set `FATIGUE_IDX` if yours is a different one).

In [ ]:
# PLACEHOLDER DATA: replace with measured amplitude increases
intensity_x = np.array([2, 3, 5, 7, 9, 10], dtype=float)           # cm
intensity_y = np.array([0.10, 0.16, 0.27, 0.36, 0.44, 0.30])        # increase in amplitude

duration_x  = np.array([3, 5, 10, 20, 40, 60, 90], dtype=float)    # s
duration_y  = np.array([0.05, 0.09, 0.17, 0.27, 0.40, 0.48, 0.33])

FATIGUE_IDX = -1   # index of the fatigue point to drop in the "cleaned" fit

## Metric
**Average L2 distance** = mean over points of the Euclidean distance between each observed point and the model's prediction at the same x. With x shared, this is the mean absolute vertical residual, `mean(|y - ŷ|)`. RMSE is printed alongside since it penalizes large misses (like the fatigue point) more.

In [ ]:
def avg_l2(y, yhat):
    return float(np.mean(np.sqrt((y - yhat) ** 2)))

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def ols(x, y):
    """Ordinary least squares y = m*x + b. Returns (m, b)."""
    A = np.column_stack([x, np.ones_like(x)])
    (m, b), *_ = np.linalg.lstsq(A, y, rcond=None)
    return m, b

def lin(x, m, b):
    return m * x + b

def expo(x, a, k):
    return a * np.exp(k * x)

def fit_exp(x, y):
    """Least-squares fit of y = a*exp(k*x). Initialised from a log-linear fit."""
    pos = y > 0
    k0, ln_a0 = np.polyfit(x[pos], np.log(y[pos]), 1)
    popt, _ = curve_fit(expo, x, y, p0=[np.exp(ln_a0), k0], maxfev=20000)
    return popt

## 1. Histograms of twitch tension

In [ ]:
groups = {"Trial 1 (voluntary)": trial_1,
          "Trial 2 (voluntary)": trial_2,
          "Trial 3 (voluntary)": trial_3,
          "Involuntary (tetanus)": involuntary}

all_vals = np.concatenate(list(groups.values()) + ([baseline] if baseline is not None else []))
bins = np.linspace(all_vals.min(), all_vals.max(), 15)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharex=True, sharey=True)
for ax, (name, vals) in zip(axes, groups.items()):
    if baseline is not None:
        ax.hist(baseline, bins=bins, color="0.75", alpha=0.7, label="baseline")
    ax.hist(vals, bins=bins, alpha=0.8, label="after warm-up")
    ax.axvline(vals.mean(), color="k", ls="--", lw=1, label=f"mean = {vals.mean():.2f}")
    ax.set_title(name)
    ax.set_xlabel("Twitch tension")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()

for name, vals in groups.items():
    print(f"{name:24s} mean = {vals.mean():.3f}  sd = {vals.std(ddof=1):.3f}  n = {len(vals)}")

## 2. Naive OLS on all points (no processing)
Expect the fit to sit **below the early/mid points** (and be pulled down at high x) because of the fatigue point.

In [ ]:
sweeps = {"Intensity (distance, cm)": (intensity_x, intensity_y),
          "Duration (s)":             (duration_x,  duration_y)}

naive = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (x, y)) in zip(axes, sweeps.items()):
    m, b = ols(x, y)
    yhat = lin(x, m, b)
    naive[name] = (m, b)
    xs = np.linspace(x.min(), x.max(), 200)
    ax.scatter(x, y, zorder=3, label="data")
    ax.plot(xs, lin(xs, m, b), "r-", label=f"OLS: y = {m:.4f}x + {b:.3f}")
    ax.set_xlabel(name); ax.set_ylabel("Increase in twitch tension")
    ax.set_title("Naive OLS, all points"); ax.legend()
    print(f"{name:26s} slope = {m:.4f}  intercept = {b:.4f}  "
          f"avg L2 = {avg_l2(y, yhat):.4f}  RMSE = {rmse(y, yhat):.4f}")
plt.tight_layout(); plt.show()

## 3. OLS with the fatigue point removed

In [ ]:
clean = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (x, y)) in zip(axes, sweeps.items()):
    keep = np.ones(len(x), bool)
    keep[FATIGUE_IDX] = False
    m, b = ols(x[keep], y[keep])
    clean[name] = (m, b)
    xs = np.linspace(x.min(), x.max(), 200)
    ax.scatter(x[keep], y[keep], zorder=3, label="data")
    ax.scatter(x[~keep], y[~keep], marker="x", c="k", zorder=3, label="fatigue point (removed)")
    ax.plot(xs, lin(xs, m, b), "g-", label=f"OLS: y = {m:.4f}x + {b:.3f}")
    ax.set_xlabel(name); ax.set_ylabel("Increase in twitch tension")
    ax.set_title("OLS without fatigue point"); ax.legend()
    l2_fit = avg_l2(y[keep], lin(x[keep], m, b))
    l2_all = avg_l2(y, lin(x, m, b))
    print(f"{name:26s} slope = {m:.4f}  intercept = {b:.4f}  "
          f"avg L2 (fit points) = {l2_fit:.4f}  RMSE = {rmse(y[keep], lin(x[keep], m, b)):.4f}  "
          f"| avg L2 (all points) = {l2_all:.4f}")
plt.tight_layout(); plt.show()

## 4. Exponential fit on all points
$y = a\,e^{kx}$ fit by nonlinear least squares to every point, fatigue point included.

In [ ]:
expfit = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (x, y)) in zip(axes, sweeps.items()):
    a, k = fit_exp(x, y)
    expfit[name] = (a, k)
    yhat = expo(x, a, k)
    xs = np.linspace(x.min(), x.max(), 200)
    ax.scatter(x, y, zorder=3, label="data")
    ax.plot(xs, expo(xs, a, k), "m-", label=f"exp: y = {a:.3f}·exp({k:.4f}x)")
    ax.set_xlabel(name); ax.set_ylabel("Increase in twitch tension")
    ax.set_title("Exponential fit, all points"); ax.legend()
    print(f"{name:26s} a = {a:.4f}  k = {k:.4f}  "
          f"avg L2 = {avg_l2(y, yhat):.4f}  RMSE = {rmse(y, yhat):.4f}")
plt.tight_layout(); plt.show()

## Summary: average L2 distance, all models (evaluated on all points)

In [ ]:
print(f"{'':26s} {'naive OLS':>11s} {'OLS no fatigue':>15s} {'exponential':>12s}")
for name, (x, y) in sweeps.items():
    l_naive = avg_l2(y, lin(x, *naive[name]))
    l_clean = avg_l2(y, lin(x, *clean[name]))
    l_exp   = avg_l2(y, expo(x, *expfit[name]))
    print(f"{name:26s} {l_naive:11.4f} {l_clean:15.4f} {l_exp:12.4f}")